## TODO convertir en cli script

In [1]:
import subprocess
import os
import pandas as pd
from pathlib import Path

def run_predivac(input_file: str, mode: str, predivac_path: str = 'predivac', allele: str = None,
                  population: str = None, mhc: str = None, pct: int = None, ppr: int = None,
                 verbosity: int = 1, outdir = "results/predivac") -> str | None:
    """
    Un wrapper de Python para ejecutar la herramienta Predivac-3.0.

    Esta función construye el comando basado en los parámetros proporcionados,
    lo ejecuta y devuelve la ruta del archivo de salida esperado si tiene éxito.

    Args:
        input_file (str): La ruta al archivo de entrada (fasta o lista de péptidos).
        mode (str): El modo de ejecución ('peptide', 'protein', 
                    'epitope_discovery', 'epitope_optimization').
        predivac_path (str, optional): La ruta al ejecutable de predivac. 
                                       Por defecto es 'predivac'.
        allele (str, optional): El alelo MHC para los modos 'peptide' y 'protein'. 
                                E.g., 'DRB1*01:01'.
        population (str, optional): La población para los modos 'epitope_discovery' 
                                    y 'epitope_optimization'. E.g., 'Chile'.
        mhc (str, optional): La clase o locus MHC. E.g., 'I', 'II', 'A,B,DRB1'.
        pct (int, optional): El umbral de cobertura poblacional (PCT) para
                             'epitope_discovery'.
        ppr (int, optional): El umbral de Peptide Percentile Rank (1-5).
        verbosity (int, optional): Nivel de verbosidad (0, 1, o 2 para -v, -vv).

    Returns:
        str | None: La ruta al archivo de salida si la ejecución fue exitosa,
                    de lo contrario, None.
    """

    if not os.path.exists(input_file):
        print(f"Error: El archivo de entrada no se encuentra en '{input_file}'")
        return None

    command = [predivac_path, input_file, mode]

    if mode in ['peptide', 'protein'] and allele:
        command.extend(['-a', allele])
    elif mode in ['epitope_discovery', 'epitope_optimization']:
        if population:
            command.extend(['--population', population])
        if mhc:
            command.extend(['--mhc', mhc])
        if mode == 'epitope_discovery' and pct is not None:
            command.extend(['--pct', str(pct)])

    if ppr is not None:
        command.extend(['-ppr', str(ppr)])
    if verbosity > 0:
        command.append('-' + 'v' * verbosity)
        
    print(f"Ejecutando comando: {' '.join(command)}")

    # 4. Determinar el nombre del archivo de salida esperado
    input_base_name = Path(input_file).stem
    output_filename = None

    if mode == 'epitope_discovery' and population and mhc and pct is not None:
        # Formato: P1_gp_epi_discovery_Chile_II_30.out
        output_filename = f"P1_{input_base_name}_epi_discovery_{population}_{mhc}_{pct}.out"
        
    elif mode == 'protein' and allele:
        # Basado en tu feedback, el formato es: 'P1_gp_prot_HLA-DRB1_0101.out'
        # para un alelo como 'DRB1*01:01'.
        # La nueva lógica añade 'HLA-', cambia '*' por '_' y elimina ':'.
        formatted_allele = "HLA-" + allele.replace('*', '_').replace(':', '')
        output_filename = f"P1_{input_base_name}_prot_{formatted_allele}.out"
        
    # else:
    #     print(f"Advertencia: No se pudo determinar el patrón del nombre de archivo de salida para el modo '{mode}' con los parámetros dados.")
    #     # Podrías añadir aquí la lógica para otros modos como 'peptide' o 'epitope_optimization'
    #     # si descubres sus patrones de nombrado.
    #     return None # Terminamos si no podemos predecir el nombre del archivo

    # 5. Ejecutar el comando (sin cambios)
    try:
        process = subprocess.run(
            command, capture_output=True, text=True, check=True
        )
        print("¡Predivac se ejecutó exitosamente!")
        print("Salida estándar (stdout):")
        print(process.stdout)

        return output_filename
        # if os.path.exists(output_filename):
        #     return output_filename
        # else:
        #     print(f"Error: La ejecución pareció exitosa, pero no se encontró el archivo de salida esperado '{output_filename}'.")
        #     print("Por favor, verifica el nombre de archivo que genera predivac.")
        #     return None

    except FileNotFoundError:
        print(f"Error: El comando '{predivac_path}' no fue encontrado.")
        return None
    except subprocess.CalledProcessError as e:
        print(f"Error al ejecutar Predivac (código de retorno {e.returncode}):")
        print(f"Stderr: {e.stderr}")
        return None

In [2]:
def parse_top(predivac_out, top_n = 1):
    
    predivac_epitope_disc = pd.read_csv(predivac_out)
    predivac_epitope_disc = predivac_epitope_disc.sort_values(by='coverage',ascending=False)
    print(predivac_epitope_disc)
    top_epitopes = predivac_epitope_disc.head(1).iloc[:,0:2].values.tolist()
    
    if not top_epitopes or len(top_epitopes) < top_n:
        raise ValueError(f"Could not select {top_n} epitopes. Found only {len(top_epitopes)}.")
    
    return top_epitopes

In [3]:
input_file = './Q76R36.fasta'       
mode = 'epitope_discovery'
population = 'Chile'
mhc = ['I','II']
threshold = 30
verbosity = 1

predivac_results = []
for m in mhc:
    predivac_out = run_predivac(input_file=input_file, mode=mode, population=population, mhc=m, pct=threshold, verbosity=verbosity)
    top_epitopes = parse_top(predivac_out, top_n=1)
    # ids
    protid = predivac_out.split("_")[1] # the fasta id comes after P1_

    id_epitopes = [(f"{protid}_E{i}_{round(p[1])}",p[0],m) for i,p in enumerate(top_epitopes,1)]
    [predivac_results.append(e) for e in id_epitopes]
    #predivac_results.append([protid,top_epitopes,m])
    
top_df = pd.DataFrame(predivac_results, columns=["id", "peptide", "mhc"]).explode("peptide")
top_df.to_csv(f"{protid}_discovery_top1.csv", index = None)

Ejecutando comando: predivac ./Q76R36.fasta epitope_discovery --population Chile --mhc I --pct 30 -v
¡Predivac se ejecutó exitosamente!
Salida estándar (stdout):
DONE

  Unnamed: 0  coverage                                            alleles
3  DPLVVAASI   67.0112  HLA-B*07:02,HLA-B*07:05,HLA-B*38:01,HLA-B*39:0...
2  AVDADDSHF   49.1691  HLA-A*23:01,HLA-A*24:02,HLA-A*24:03,HLA-A*25:0...
0  RLFFKCIYR   39.7457  HLA-A*01:01,HLA-A*03:01,HLA-A*68:01,HLA-A*68:1...
1  SIIGILHLI   32.3329    HLA-A*02:01,HLA-A*02:06,HLA-A*02:11,HLA-A*02:22
4  ILHLILWIL   30.5040    HLA-B*35:02,HLA-B*35:09,HLA-B*35:12,HLA-B*56:02
Ejecutando comando: predivac ./Q76R36.fasta epitope_discovery --population Chile --mhc II --pct 30 -v
¡Predivac se ejecutó exitosamente!
Salida estándar (stdout):
DONE

  Unnamed: 0  coverage                                            alleles
0  LVVAASIIG   30.7443  HLA-DRB1*04:01,HLA-DRB1*04:02,HLA-DRB1*04:03,H...


### generate linkers

In [6]:
import random
def generate_linkers(length: int, total: int, seed = 42):
    """Generates a list of random linkers, excluding Cysteine."""
    print(f"INFO: Generating {total} random linkers of length {length}...")
    random.seed(seed)

    AMINO_ACIDS = "ADEFGHIKLMNPQRSTVWY"
    linkers = ["".join(random.choices(AMINO_ACIDS, k = length)) for _ in range(total)]
    return linkers

generate_linkers(4, 1000, 42)

INFO: Generating 1000 random linkers of length 4...


['QAHG',
 'RQVD',
 'LAGM',
 'AFQN',
 'GPTA',
 'TRIE',
 'YIDD',
 'VPTR',
 'NYKN',
 'TPVN',
 'RAGH',
 'DGDH',
 'QIKF',
 'HWQP',
 'FRFK',
 'YQNR',
 'VSGA',
 'HHGW',
 'VHQK',
 'WLHG',
 'NGPW',
 'KGYM',
 'DAEP',
 'TLDK',
 'YNYV',
 'ARQN',
 'HQEL',
 'LYVH',
 'MFWV',
 'HQPE',
 'SNSN',
 'AIAW',
 'VTHD',
 'VWDM',
 'DSSE',
 'MNHV',
 'LGNR',
 'FHYQ',
 'LMEG',
 'IPGG',
 'DPGW',
 'VDGQ',
 'GEWN',
 'LSTF',
 'DLLL',
 'RQYD',
 'KIVG',
 'FLLH',
 'GWLV',
 'NAYT',
 'YWVF',
 'MGKD',
 'KYHS',
 'LLYY',
 'NREH',
 'YPNS',
 'DPMV',
 'EYDF',
 'PQGE',
 'VGPP',
 'KPMW',
 'FRGK',
 'QHIS',
 'DLYY',
 'DGHW',
 'VVKE',
 'TRPY',
 'QATH',
 'QWEE',
 'ENHP',
 'RFQH',
 'MWVD',
 'LHAS',
 'QGSN',
 'LADV',
 'WNTP',
 'EEHW',
 'TVWF',
 'GDSV',
 'KPEW',
 'VYTV',
 'ARIW',
 'TVTH',
 'SEVV',
 'GTLH',
 'TGAF',
 'IVYH',
 'QKYN',
 'WEYF',
 'YHEL',
 'RHPM',
 'KNGR',
 'AWNR',
 'SQID',
 'QIHV',
 'RHHK',
 'KHEK',
 'WQWP',
 'HNAH',
 'LPQL',
 'LGLW',
 'TFDM',
 'QITS',
 'QGFA',
 'GMVD',
 'KPFR',
 'MGQA',
 'SSEL',
 'FYMA',
 'GVLT',
 'QYPY',
 

In [34]:
import itertools
def generate_constructs(epitopes, linkers, structure, epitopes_must_be_different = True, linkers_must_be_different = False):
    """
    Generates vaccine constructs for any given architecture in a memory-efficient way.

    This function is a generator that yields tuples of (sequence_id, sequence_string).

    Args:
        epitopes: A list of epitope sequences for assembly.
        linkers: A list of linker sequences for assembly.
        structure: A string defining the construct architecture (e.g., 'LELE', 'ELE').
        epitopes_must_be_different: If True, epitopes cannot be repeated.
        linkers_must_be_different: If True, linkers cannot be repeated.

    Yields:
        A tuple containing a unique sequence ID and the full vaccine sequence string.
    """
    if not epitopes or not linkers:
        return

    structure = structure.upper()
    num_e = structure.count('E')
    num_l = structure.count('L')

    # --- 1. Create the appropriate iterator for each component ---
    
    # Epitope Iterator
    if num_e > 0:
        if epitopes_must_be_different:
            if len(epitopes) < num_e:
                raise ValueError(f"Not enough unique epitopes for structure '{structure}'.")
            epitope_iter = itertools.permutations(epitopes, num_e)
        else:
            epitope_iter = itertools.product(epitopes, repeat=num_e)
    else:
        epitope_iter = [()] # If no epitopes, provide one empty tuple to iterate over

    # Linker Iterator
    if num_l > 0:
        if linkers_must_be_different:
            if len(linkers) < num_l:
                raise ValueError(f"Not enough unique linkers for structure '{structure}'.")
            linker_iter = itertools.permutations(linkers, num_l)
        else:
            linker_iter = itertools.product(linkers, repeat=num_l)
    else:
        linker_iter = [()] # If no linkers, provide one empty tuple

    # --- 2. Combine the component iterators and assemble the constructs ---
    
    i = 0
    # The main product combines all possible epitope sets with all possible linker sets
    for epitope_tuple, linker_tuple in itertools.product(epitope_iter, linker_iter):
        e_idx, l_idx = 0, 0
        result_sequence = []
        
        # Weave the epitopes and linkers together based on the structure string
        for component_type in structure:
            if component_type == 'E':
                result_sequence.append(epitope_tuple[e_idx])
                e_idx += 1
            elif component_type == 'L':
                result_sequence.append(linker_tuple[l_idx])
                l_idx += 1
        
        # Format the output ID and sequence
        epitope_id_part = "-".join(epitope_tuple)
        seq_id = f"{i+1}|{epitope_id_part}"
        sequence = "".join(result_sequence)
        i += 1
        
        yield seq_id, sequence

In [35]:
linker_list = generate_linkers(4, 1000, 42)
peptide_list = top_df["peptide"].tolist()
print(peptide_list)
construct_list = list(generate_constructs(peptide_list, linker_list, "ELE", epitopes_must_be_different = True))

INFO: Generating 1000 random linkers of length 4...
['DPLVVAASI', 'LVVAASIIG']


In [50]:
6000*22

132000

In [52]:
5*3000*25

375000

In [42]:
len('DPLVVAASIKEDMLVVAASIIG')

22

In [22]:
linker_list[:5]

['QAHG', 'RQVD', 'LAGM', 'AFQN', 'GPTA']

In [18]:
construct_list[800]

('construct_801|DPLVVAASI-LVVAASIIG', 'DPLVVAASIQQHKLVVAASIIG')

In [9]:
import random

#random.seed(12)
# without allowing cystein which can form disulfide bonds 
def generate_seq(k):
    AMINOACIDS = "ADEFGHIKLMNPQRSTVWY" 
    sequence = "".join(random.choices(AMINOACIDS, k=k))

    return sequence

def generate_linkers(k, total=1000):
    random.seed(42)
    
    linkers = []
    for n in range(total):
        
        linker = generate_seq(k)
        linkers.append(linker)
    return linkers

In [10]:
N = 1000

linkers = generate_linkers(4, total=N)

In [11]:
import math

def calculate_combinations(
    num_epitopes: int,
    num_linkers: int,
    structure: str = 'LELE',
    epitopes_must_be_different: bool = True
) -> int:
    """
    Calculates the total number of possible vaccine constructs for a given structure.

    Args:
        num_epitopes: The number of epitopes in the pool.
        num_linkers: The number of linkers in the pool.
        structure: A string defining the construct architecture, e.g., 'LE' or 'LELE'.
        epitopes_must_be_different: If True, epitopes cannot be repeated.
                                    (Only applies to structures with >1 epitope).

    Returns:
        The total number of unique combinations for the specified structure.
    """
    if num_epitopes == 0 or num_linkers == 0:
        return 0

    if structure == 'LE':
        # Simple multiplication for one linker and one epitope
        return num_linkers * num_epitopes
        
    elif structure == 'LELE':
        # Linkers can be repeated
        linker_combinations = num_linkers ** 2
        
        # Epitope combinations
        if epitopes_must_be_different:
            if num_epitopes < 2: return 0
            # Permutation of epitopes: P(n, 2)
            epitope_combinations = math.perm(num_epitopes, 2)
        else:
            # Epitopes can be repeated
            epitope_combinations = num_epitopes ** 2
            
        return linker_combinations * epitope_combinations
    elif structure == 'ELE':
        linker_combinations = num_linkers
        if epitopes_must_be_different:
            if num_epitopes < 2: return 0
            epitope_combinations = math.perm(num_epitopes, 2)
        else:
            epitope_combinations = num_epitopes ** 2
        return linker_combinations * epitope_combinations
        
    else:
        raise ValueError(f"Structure '{structure}' is not supported.")

# --- Ejemplo de Uso ---
# Caso 1: Nuevo constructo [L-E]
total_le = calculate_combinations(2, 1000, structure='LE')
print(f"Número total de constructos [L-E]: {total_le:,}")

# Caso 2: Constructo original [L-E-L-E]
total_lele = calculate_combinations(2, 400, structure='ELE', epitopes_must_be_different=True)
print(f"Número total de constructos [L-E-L-E]: {total_lele:,}")
# caso 3
total_ele = calculate_combinations(2, 1000, structure='ELE')
print(f"Número total de constructos [E-L-E]: {total_ele:,}")

Número total de constructos [L-E]: 2,000
Número total de constructos [L-E-L-E]: 800
Número total de constructos [E-L-E]: 2,000


9000000